# Making API call

```python
from playwright.sync_api import *

def test_users_api(page: Page):
    response = page.goto("https://dummyjson.com/users/1")
    user_data = response.json()
    print(user_data)
    assert "firstName" in user_data
    assert "lasttName" in user_data
```

# API Request Context

**Syntax**:

```python
def new_context(self,
                *,
                base_url: str | None = None,
                extra_http_headers: dict[str, str] | None = None,
                http_credentials: HttpCredentials | None = None,
                ignore_https_errors: bool | None = None,
                proxy: ProxySettings | None = None,
                user_agent: str | None = None,
                timeout: float | None = None,
                storage_state: StorageState | str | Path | None = None,
                client_certificates: list[ClientCertificate] | None = None,
                fail_on_status_code: bool | None = None) -> APIRequestContext
```

**Example**:

```python
from playwright.sync_api import *

def test_users_api(playwright: Playwright):

    api_context = playwright.request.new_context()
    
    response = api_context.get("https://dummyjson.com/users/1")
    user_data = response.json()
    print(user_data)
    assert "firstName" in user_data
    assert "lasttName" in user_data
    api_context.dispose()

def test_users_api_with_params(playwright: Playwright):

    api_context = playwright.request.new_context(
        base_url="https://dummyjson.com"
    )
    response_user = api_context.get("/users/1")
    response_products = api_context.get("/products")
    api_context.dispose()
```

# API Query String

```python
import pytest
from playwright.sync_api import *

@pytest.fixture
def myapi_context(playwright: Playwright):
    api_context = playwright.request.new_context(
        base_url="https://dummyjson.com"
    )
    yield  api_context
    api_context.dispose()

def test_users_api(myapi_context: APIRequestContext):
    query="John"
    response_user = myapi_context.get(f"/users/search?q={query}")
    users_data = response_user.json()
    print("Users found: ", users_data["total"])

    for user in users_data["users"]:
        print("Checking user: ", user["firstName"])
        assert query in user["firstName"]
```

# CURD Operations

```python

import pytest
from playwright.sync_api import *

@pytest.fixture
def myapi_context(playwright: Playwright):
    api_context = playwright.request.new_context(
        base_url="https://dummyjson.com",
        extra_http_headers={'Content-Type': 'application/json'},
    )
    yield  api_context
    api_context.dispose()

# POST REQUEST
def test_create_user(myapi_context: APIRequestContext):
    response = myapi_context.post(
        url="/users/add",
        headers={'Content-Type': 'application/json'},
        data={
            "firstName": "Daimen",
            "lastName": "Smith",
            "age": 27
        }
    )

    user_data = response.json()
    print("\n", user_data, "\n")

    assert user_data["firstName"] == "Daimen"


def test_update_user(api_context: APIRequestContext):
    print(
        "Default Last Name:",
        api_context.get("users/1").json()["lastName"]
    )

    response = api_context.put(
        "users/1",
        data={
            "lastName": "Smith",
            "age": 20,
        }
    )
    user_data = response.json()

    assert user_data["lastName"] == "Smith"
    assert user_data["age"] == 20


def test_remove_user(api_context: APIRequestContext):
    response = api_context.delete("users/1")

    user_data = response.json()

    assert user_data["isDeleted"]

```

# Mock API with custom data

```python
from playwright.sync_api import *


def on_api_call(route: Route):
    response = route.fetch()
    user_data = response.json()

    user_data["lastName"] = "Smith"
    user_data["age"] = 20

    route.fulfill(
        response=response,
        json=user_data,
    )


def test_user_api(page: Page):
    USERS_API_URL = "https://dummyjson.com/users/1"

    page.route(USERS_API_URL, on_api_call)

    response = page.goto(USERS_API_URL)
    print("Got data:", response.json())
```